# Molecular Fingerprinting & Similarity Analysis Using RDKit

Independent educational portfolio project. Compare example molecules using Morgan fingerprints, MACCS keys and Tanimoto similarity.

**Caution:** fingerprint similarity is not proof of biological activity or therapeutic equivalence.

## 1. Install dependencies

In [ ]:
!pip install -q rdkit pandas numpy matplotlib seaborn reportlab


## 2. Imports and setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import AllChem, MACCSkeys, Draw
from IPython.display import display
from google.colab import files

print("RDKit version:", rdBase.rdkitVersion)
OUT = Path("output")
OUT.mkdir(exist_ok=True)


## 3. Load the sample dataset

In [ ]:
data_path = Path("data/molecule_dataset.csv")
if not data_path.exists():
    uploaded = files.upload()
    data_path = Path(next(iter(uploaded)))
df = pd.read_csv(data_path)
display(df.head())
print("Dataset rows:", len(df))


## 4. Validate SMILES

In [ ]:
records, invalid = [], []
for _, row in df.iterrows():
    mol = Chem.MolFromSmiles(str(row["SMILES"]))
    if mol is None:
        invalid.append({"Name": row["Name"], "SMILES": row["SMILES"]})
    else:
        records.append({"Name": str(row["Name"]), "SMILES": str(row["SMILES"]), "mol": mol})
if len(records) < 2:
    raise ValueError("At least two valid molecules are required.")
names = [r["Name"] for r in records]
mols = [r["mol"] for r in records]
print("Valid:", len(records), "| Invalid:", len(invalid))


## 5. Generate Morgan fingerprints and MACCS keys

In [ ]:
RADIUS, N_BITS = 2, 2048
morgan = [AllChem.GetMorganFingerprintAsBitVect(m, RADIUS, nBits=N_BITS) for m in mols]
maccs = [MACCSkeys.GenMACCSKeys(m) for m in mols]
print("Fingerprints generated for", len(mols), "molecules")


## 6. Calculate Tanimoto similarity matrices

In [ ]:
def similarity_matrix(fps):
    return np.array([DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps])
morgan_matrix = similarity_matrix(morgan)
maccs_matrix = similarity_matrix(maccs)
morgan_df = pd.DataFrame(morgan_matrix, index=names, columns=names)
maccs_df = pd.DataFrame(maccs_matrix, index=names, columns=names)
display(morgan_df.round(3))


## 7. Plot heatmaps

In [ ]:
for label, frame, cmap in [("Morgan", morgan_df, "viridis"), ("MACCS", maccs_df, "mako")]:
    plt.figure(figsize=(12, 10))
    sns.heatmap(frame, cmap=cmap, vmin=0, vmax=1, square=True)
    plt.title(f"{label} Fingerprint Tanimoto Similarity")
    plt.xticks(rotation=75, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUT / f"{label.lower()}_similarity_heatmap.png", dpi=300)
    plt.show()


## 8. Rank pairwise similarities

In [ ]:
pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pairs.append([names[i], names[j], morgan_matrix[i, j], maccs_matrix[i, j]])
pair_df = pd.DataFrame(pairs, columns=["Molecule_1", "Molecule_2", "Morgan_Tanimoto", "MACCS_Tanimoto"])
pair_df = pair_df.sort_values("Morgan_Tanimoto", ascending=False)
display(pair_df.head(15).round(3))


## 9. Draw molecule structures

In [ ]:
img = Draw.MolsToGridImage(mols[:20], molsPerRow=4, subImgSize=(250, 200), legends=names[:20], useSVG=False)
display(img)
img.save(str(OUT / "molecule_structure_grid.png"))


## 10. Export CSV files

In [ ]:
morgan_df.to_csv(OUT / "morgan_similarity_matrix.csv")
maccs_df.to_csv(OUT / "maccs_similarity_matrix.csv")
pair_df.to_csv(OUT / "pairwise_similarity_scores.csv", index=False)
pd.DataFrame({
    "Name": names,
    "Canonical_SMILES": [Chem.MolToSmiles(m) for m in mols],
    "Morgan_Bits": [N_BITS] * len(mols),
    "Morgan_On_Bits": [fp.GetNumOnBits() for fp in morgan],
    "MACCS_On_Bits": [fp.GetNumOnBits() for fp in maccs]
}).to_csv(OUT / "fingerprint_summary.csv", index=False)
if invalid:
    pd.DataFrame(invalid).to_csv(OUT / "invalid_smiles_rows.csv", index=False)
print("CSV outputs saved.")


## 11. Generate a PDF report

In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image

styles = getSampleStyleSheet()
pdf_path = OUT / "Molecular_Similarity_Analysis_Report.pdf"
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
story = [
    Paragraph("Molecular Fingerprinting & Similarity Analysis", styles["Title"]),
    Spacer(1, 10),
    Paragraph(f"Valid molecules analysed: {len(records)}; invalid SMILES skipped: {len(invalid)}.", styles["BodyText"]),
    Spacer(1, 10),
    Paragraph("Top 10 pairs by Morgan Tanimoto similarity", styles["Heading2"])
]
table_data = [["Molecule 1", "Molecule 2", "Morgan Tanimoto"]]
for _, r in pair_df.head(10).iterrows():
    table_data.append([r.Molecule_1, r.Molecule_2, f"{r.Morgan_Tanimoto:.3f}"])
table = Table(table_data, repeatRows=1, colWidths=[145, 145, 100])
table.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#263B5A")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("GRID", (0,0), (-1,-1), 0.4, colors.grey)
]))
story += [table, Spacer(1, 12), Image(str(OUT / "morgan_similarity_heatmap.png"), width=460, height=380),
          Spacer(1, 8), Paragraph("Fingerprint similarity is not evidence of biological activity, safety, potency or therapeutic equivalence.", styles["BodyText"])]
doc.build(story)
print("PDF saved:", pdf_path)


## 12. Download all generated outputs

In [ ]:
import shutil
zip_path = shutil.make_archive("Molecular_Similarity_Analysis_Outputs", "zip", OUT)
files.download(zip_path)


## 13. Interpretation and limitations

- Tanimoto similarity ranges from 0 to 1 for these binary fingerprints.
- A score of 1 indicates identical fingerprint bit vectors.
- Rankings can change with fingerprint type and parameters.
- Similarity does not establish biological activity, safety, potency, binding affinity or clinical usefulness.